In [1]:
import torch 
from model import *

In [2]:
if torch.cuda.is_available():
    device = torch.device('cuda:0')
    print('The code uses GPU...')
else:
    device = torch.device('cpu')
    print('The code uses CPU!!!')

protein_dim = 100
atom_dim = 34
hid_dim = 64
n_layers = 3
n_heads = 8
pf_dim = 256
dropout = 0.1
batch = 64
lr = 1e-4
weight_decay = 1e-4
decay_interval = 5
lr_decay = 1.0
iteration = 100
kernel_size = 7

The code uses GPU...


In [3]:
encoders = [Encoder(protein_dim, hid_dim, n_layers, kernel_size, dropout,
                      device) for _ in range(7)]
decoders = [Decoder(atom_dim, hid_dim, n_layers, n_heads, pf_dim,
                      DecoderLayer, SelfAttention, PositionwiseFeedforward,
                      dropout, device) for _ in range(7)]

predictor_list = [Predictor(e, d, device) for e,d in zip(encoders,decoders)]
model = Predictors(predictor_list)

In [4]:
model

Predictors(
  (predictors1): Predictor(
    (encoder): Encoder(
      (convs): ModuleList(
        (0): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
        (1): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
        (2): Conv1d(64, 128, kernel_size=(7,), stride=(1,), padding=(3,))
      )
      (dropout): Dropout(p=0.1, inplace=False)
      (fc): Linear(in_features=100, out_features=64, bias=True)
      (gn): GroupNorm(8, 128, eps=1e-05, affine=True)
      (ln): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
    )
    (decoder): Decoder(
      (ln): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
      (sa): SelfAttention(
        (w_q): Linear(in_features=64, out_features=64, bias=True)
        (w_k): Linear(in_features=64, out_features=64, bias=True)
        (w_v): Linear(in_features=64, out_features=64, bias=True)
        (fc): Linear(in_features=64, out_features=64, bias=True)
        (do): Dropout(p=0.1, inplace=False)
      )
      (layer